In [ ]:
# ============================================================
# 1. INSTALL REQUIRED LIBRARIES
# This installs TensorFlow and Matplotlib
# ============================================================
!pip install tensorflow matplotlib


# ============================================================
# 2. IMPORT REQUIRED LIBRARIES
# TensorFlow for deep learning
# Matplotlib for displaying images
# NumPy and os for basic operations
# ============================================================
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt
import numpy as np
import os


# ============================================================
# 3. DOWNLOAD AND EXTRACT THE HORSE2ZEBRA DATASET
# Removes old dataset files if already present
# Downloads the dataset
# Extracts it automatically
# ============================================================
!rm -rf horse2zebra
!rm -f horse2zebra.zip horse2zebra.zip.*
!wget -O horse2zebra.zip https://efrosgans.eecs.berkeley.edu/cyclegan/datasets/horse2zebra.zip
!unzip -oq horse2zebra.zip


# ============================================================
# 4. SET IMAGE PARAMETERS
# Defines image height, width, batch size
# AUTOTUNE helps TensorFlow optimize performance
# ============================================================
IMG_HEIGHT = 128
IMG_WIDTH = 128
BATCH_SIZE = 1
AUTOTUNE = tf.data.AUTOTUNE


# ============================================================
# 5. LOAD DATASET PATHS
# These paths point to training and testing images
# trainA = horse images
# trainB = zebra images
# ============================================================
trainA_path = "horse2zebra/trainA/*.jpg"
trainB_path = "horse2zebra/trainB/*.jpg"
testA_path = "horse2zebra/testA/*.jpg"
testB_path = "horse2zebra/testB/*.jpg"

trainA = tf.data.Dataset.list_files(trainA_path)
trainB = tf.data.Dataset.list_files(trainB_path)
testA = tf.data.Dataset.list_files(testA_path)
testB = tf.data.Dataset.list_files(testB_path)


# ============================================================
# 6. PREPROCESS IMAGES
# Reads images
# Converts them into RGB format
# Resizes them to 128x128
# Normalizes pixel values to range [-1, 1]
# ============================================================
def load_image(image_file):
    image = tf.io.read_file(image_file)
    image = tf.image.decode_jpeg(image, channels=3)   # Force RGB
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    image = (tf.cast(image, tf.float32) / 127.5) - 1
    return image

trainA = trainA.map(load_image, num_parallel_calls=AUTOTUNE).shuffle(1000).batch(BATCH_SIZE)
trainB = trainB.map(load_image, num_parallel_calls=AUTOTUNE).shuffle(1000).batch(BATCH_SIZE)
testA = testA.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)
testB = testB.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)


# ============================================================
# 7. BUILD GENERATOR MODEL
# Generator converts:
# Horse -> Zebra OR Zebra -> Horse
# ============================================================
def build_generator():
    inputs = layers.Input(shape=[IMG_HEIGHT, IMG_WIDTH, 3])

    x = layers.Conv2D(64, 7, padding='same')(inputs)
    x = layers.ReLU()(x)

    x = layers.Conv2D(128, 3, strides=2, padding='same')(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(256, 3, strides=2, padding='same')(x)
    x = layers.ReLU()(x)

    x = layers.Conv2DTranspose(128, 3, strides=2, padding='same')(x)
    x = layers.ReLU()(x)

    x = layers.Conv2DTranspose(64, 3, strides=2, padding='same')(x)
    x = layers.ReLU()(x)

    outputs = layers.Conv2D(3, 7, padding='same', activation='tanh')(x)

    return Model(inputs, outputs)


# ============================================================
# 8. BUILD DISCRIMINATOR MODEL
# Discriminator checks whether image is:
# Real or Fake
# ============================================================
def build_discriminator():
    inputs = layers.Input(shape=[IMG_HEIGHT, IMG_WIDTH, 3])

    x = layers.Conv2D(64, 4, strides=2, padding='same')(inputs)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(128, 4, strides=2, padding='same')(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(256, 4, strides=2, padding='same')(x)
    x = layers.LeakyReLU(0.2)(x)

    outputs = layers.Conv2D(1, 4, padding='same')(x)

    return Model(inputs, outputs)


# ============================================================
# 9. CREATE CYCLEGAN MODELS
# G_AB = Horse -> Zebra
# G_BA = Zebra -> Horse
# D_A = Horse Discriminator
# D_B = Zebra Discriminator
# ============================================================
G_AB = build_generator()
G_BA = build_generator()

D_A = build_discriminator()
D_B = build_discriminator()


# ============================================================
# 10. DEFINE LOSS FUNCTIONS
# Generator loss -> helps generator fool discriminator
# Discriminator loss -> helps discriminator detect fake images
# Cycle loss -> ensures image remains consistent after translation
# ============================================================
loss_obj = tf.keras.losses.MeanSquaredError()

def generator_loss(fake):
    return loss_obj(tf.ones_like(fake), fake)

def discriminator_loss(real, fake):
    real_loss = loss_obj(tf.ones_like(real), real)
    fake_loss = loss_obj(tf.zeros_like(fake), fake)
    return (real_loss + fake_loss) * 0.5

def cycle_loss(real_image, cycled_image):
    return tf.reduce_mean(tf.abs(real_image - cycled_image))


# ============================================================
# 11. DEFINE OPTIMIZERS
# Separate optimizers are used for each model
# ============================================================
G_AB_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
G_BA_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
D_A_optimizer  = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
D_B_optimizer  = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)


# ============================================================
# 12. DEFINE TRAINING STEP
# This is the core of CycleGAN training
# It performs:
# A -> B -> A
# B -> A -> B
# Calculates generator, discriminator and cycle losses
# Updates all four models
# ============================================================
@tf.function
def train_step(real_A, real_B):
    with tf.GradientTape(persistent=True) as tape:
        # A -> B -> A
        fake_B = G_AB(real_A, training=True)
        cycled_A = G_BA(fake_B, training=True)

        # B -> A -> B
        fake_A = G_BA(real_B, training=True)
        cycled_B = G_AB(fake_A, training=True)

        # Discriminator outputs
        disc_real_A = D_A(real_A, training=True)
        disc_fake_A = D_A(fake_A, training=True)

        disc_real_B = D_B(real_B, training=True)
        disc_fake_B = D_B(fake_B, training=True)

        # Generator losses
        gen_AB_loss = generator_loss(disc_fake_B)
        gen_BA_loss = generator_loss(disc_fake_A)

        # Cycle consistency loss
        total_cycle_loss = cycle_loss(real_A, cycled_A) + cycle_loss(real_B, cycled_B)

        # Total generator losses
        total_gen_AB_loss = gen_AB_loss + 10 * total_cycle_loss
        total_gen_BA_loss = gen_BA_loss + 10 * total_cycle_loss

        # Discriminator losses
        disc_A_loss = discriminator_loss(disc_real_A, disc_fake_A)
        disc_B_loss = discriminator_loss(disc_real_B, disc_fake_B)

    # Compute gradients
    grads_G_AB = tape.gradient(total_gen_AB_loss, G_AB.trainable_variables)
    grads_G_BA = tape.gradient(total_gen_BA_loss, G_BA.trainable_variables)

    grads_D_A = tape.gradient(disc_A_loss, D_A.trainable_variables)
    grads_D_B = tape.gradient(disc_B_loss, D_B.trainable_variables)

    # Apply gradients
    G_AB_optimizer.apply_gradients(zip(grads_G_AB, G_AB.trainable_variables))
    G_BA_optimizer.apply_gradients(zip(grads_G_BA, G_BA.trainable_variables))

    D_A_optimizer.apply_gradients(zip(grads_D_A, D_A.trainable_variables))
    D_B_optimizer.apply_gradients(zip(grads_D_B, D_B.trainable_variables))

    return total_gen_AB_loss, total_gen_BA_loss, disc_A_loss, disc_B_loss


# ============================================================
# 13. TRAIN THE MODEL
# Runs the training loop for multiple epochs
# Displays loss values every 100 steps
# ============================================================
EPOCHS = 3   # You can change to 5 or 10 for better output

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    for i, (real_A, real_B) in enumerate(tf.data.Dataset.zip((trainA, trainB))):
        g_ab_loss, g_ba_loss, d_a_loss, d_b_loss = train_step(real_A, real_B)

        if i % 100 == 0:
            print(f"Step {i}: G_AB={g_ab_loss:.4f}, G_BA={g_ba_loss:.4f}, D_A={d_a_loss:.4f}, D_B={d_b_loss:.4f}")


# ============================================================
# 14. FUNCTION TO DISPLAY ORIGINAL AND TRANSLATED IMAGES
# Converts images back to normal display format
# Shows original and translated images side by side
# ============================================================
def display_images(original, translated):
    original = (original[0] + 1) / 2.0
    translated = (translated[0] + 1) / 2.0

    plt.figure(figsize=(8,4))

    plt.subplot(1,2,1)
    plt.imshow(original)
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(translated)
    plt.title("Translated Image")
    plt.axis("off")

    plt.show()


# ============================================================
# 15. TEST HORSE -> ZEBRA TRANSLATION
# Takes one horse image and converts it into zebra style
# ============================================================
for sample in testA.take(1):
    translated = G_AB(sample, training=False)
    display_images(sample, translated)


# ============================================================
# 16. TEST ZEBRA -> HORSE TRANSLATION
# Takes one zebra image and converts it into horse style
# ============================================================
for sample in testB.take(1):
    translated = G_BA(sample, training=False)
    display_images(sample, translated)

--2026-04-01 10:11:12--  https://efrosgans.eecs.berkeley.edu/cyclegan/datasets/horse2zebra.zip
Resolving efrosgans.eecs.berkeley.edu (efrosgans.eecs.berkeley.edu)... 128.32.244.190
Connecting to efrosgans.eecs.berkeley.edu (efrosgans.eecs.berkeley.edu)|128.32.244.190|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 116867962 (111M) [application/zip]
Saving to: ‘horse2zebra.zip’

horse2zebra.zip     100%[===================>] 111.45M  41.6MB/s    in 2.7s    

2026-04-01 10:11:16 (41.6 MB/s) - ‘horse2zebra.zip’ saved [116867962/116867962]


Epoch 1/3
Step 0: G_AB=6.3920, G_BA=6.3921, D_A=0.5127, D_B=0.4823
Step 100: G_AB=4.3636, G_BA=4.4194, D_A=0.1926, D_B=0.1818
Step 200: G_AB=4.7690, G_BA=4.7232, D_A=0.1036, D_B=0.1497
Step 300: G_AB=4.1998, G_BA=4.2247, D_A=0.1349, D_B=0.2188
Step 400: G_AB=5.2631, G_BA=4.9056, D_A=0.2845, D_B=0.1381
Step 500: G_AB=3.4806, G_BA=3.1783, D_A=0.3342, D_B=0.1523
Step 600: G_AB=3.1940, G_BA=3.0367, D_A=0.2544, D_B=0.2475
Step 700: